# Caso Práctico Unidad 1

## Ejercicio 2

### Lectura de datos



In [2]:
import datetime
import pandas as pd
import plotly.express as px

data_path = "AirPassengers.csv"

df = pd.read_csv(data_path)
months = []

for row in range(df.shape[0]):
    months.append(datetime.datetime.strptime(df['Month'].iloc[row], '%Y-%m'))

df['Fecha'] = months
df.set_index('Fecha', inplace=True, drop=True)
df.head()

,Month,#Passengers
Fecha,,
1949-01-01,1949-01,112
1949-02-01,1949-02,118
1949-03-01,1949-03,132
1949-04-01,1949-04,129
1949-05-01,1949-05,121


### Visualización de la serie temporal

In [3]:
fig = px.line(df,
              x=df.index,
              y=df['#Passengers'],
              markers=False,
              labels="",
              title="Pasajeros AirPassengers")
fig['layout']['xaxis1']['title'] = ''
fig['layout']['yaxis1']['title'] = 'Número de pasajeros'

fig.show()

La serie presenta una tendencia creciente clara a lo largo del tiempo. Además, se observan patrones repetitivos que indican la presencia de estacionalidad. La amplitud de las fluctuaciones aumenta con el tiempo, lo que sugiere que la variabilidad no es constante.

### Comprobacion de valores nulo

In [4]:
missing_values_mask = df.isnull()


fig = px.imshow(missing_values_mask.T,
                labels=dict(x="Time Index", y="Variables", color="Missing Values"),
                x=df.index,
                y=df.columns,
                color_continuous_scale=[[0, 'green'], [1, 'white']])

fig.update_layout(title="Valores nulos",
                  xaxis_title="",
                  yaxis_title="Variables",
                  xaxis_showgrid=True, yaxis_showgrid=True)


fig.show()

No se observan valores nulos en el conjunto de datos, lo que indica que la serie está completa y no requiere ningún tratamiento previo de imputación o eliminación de datos. Esto permite trabajar directamente con la serie sin necesidad de aplicar técnicas adicionales de limpieza.

### Aplicación de suavizado a la serie temporal y visualización de la serie resultante

In [5]:
df["rolling_monthly_avg"] = df["#Passengers"].rolling(window=5).mean()


fig = px.line(df,
              x=df.index,
              y=df.rolling_monthly_avg,
              markers=False,
              labels="fdsf",
              title="Pasajeros suavizados con MA")
fig['layout']['xaxis1']['title'] = ''
fig['layout']['yaxis1']['title'] = 'Pasajeros Suavizados con MA'

fig.show()

El suavizado mediante media móvil permite observar con mayor claridad la tendencia general de la serie, reduciendo las fluctuaciones a corto plazo. En la gráfica se aprecia que la tendencia creciente se mantiene de forma más clara, mientras que las variaciones estacionales aparecen más suavizadas. Esto facilita la interpretación del comportamiento global de la serie, eliminando parte del ruido presente en los datos originales.

### Visualización de la distribución de datos

In [6]:
fig = px.histogram(df['#Passengers'])
fig.show()

In [7]:
fig = px.histogram(df['rolling_monthly_avg'])
fig.show()

In [8]:
df['Passengers Differentiation'] = df['#Passengers'].diff(periods=1)

fig = px.histogram(df['Passengers Differentiation'])
fig.show()

La distribución de los valores originales muestra una mayor dispersión, con una concentración de datos en valores medios pero con una cola hacia valores más altos, lo que refleja la tendencia creciente de la serie. En el caso de la serie suavizada, la distribución es más compacta, ya que el efecto del suavizado reduce la variabilidad y elimina parte de las fluctuaciones extremas. Por último, la serie diferenciada presenta una distribución centrada alrededor de cero, lo que indica que la tendencia ha sido eliminada en gran medida y que la serie se aproxima a un comportamiento más estacionario.

### Generación de gráficos estacionales

In [12]:
df['Month_Name'] = df.index
df['Month_Name'] = pd.to_datetime(df['Month_Name'])
df['Month_Name'] = df['Month_Name'].dt.strftime('%b')

df['Year'] = df.index.year

In [10]:
plot_df = df.groupby(['Year', 'Month_Name'])['#Passengers'].mean().reset_index().dropna()
fig = px.box(plot_df, y="#Passengers", x="Month_Name", log_y=True, title="Box Plot: Month Passengers Averages")

fig.show()

In [11]:
interested_years = [1949, 1950, 1951]
plot_df = pd.pivot_table(df[df.Year.isin(interested_years)], index="Year", values='#Passengers', columns="Month_Name", aggfunc="mean")
plot_df.index = "Y" + plot_df.index.astype(str)

fig = px.imshow(plot_df, height=600, title="Pasajeros: Mes vs Año")
fig['layout']['xaxis1']['title'] = 'Mes'
fig['layout']['yaxis1']['title'] = 'Año'
fig.show()

En los gráficos estacionales se observa claramente que el comportamiento de la serie depende del mes del año. El boxplot muestra que algunos meses, como verano y principios de otoño, presentan valores más altos de pasajeros y mayor variabilidad, mientras que otros meses tienen valores más bajos y más estables. Por otro lado, el mapa de calor permite ver cómo estos patrones se repiten a lo largo de los años, confirmando la presencia de una estacionalidad marcada. Además, se aprecia que, independientemente del mes, los valores tienden a aumentar con el paso de los años, lo que refuerza la idea de una tendencia creciente en la serie.

### Aplicación de la técnica de descomposición de Series Temporales

In [15]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def decomposition_plot(
        ts_index, observed=None, seasonal=None, trend=None, resid=None
    ):
        """Plots the decomposition output
        """
        series = []
        if observed is not None:
            series += ["Original"]
        if trend is not None:
            series += ["Trend"]
        if seasonal is not None:
            series += ["Seasonal"]
        if resid is not None:
            series += ["Residual"]
        if len(series) == 0:
            raise ValueError(
                "All component flags were off. Need atleast one of the flags turned on to plot."
            )
        fig = make_subplots(
            rows=len(series), cols=1, shared_xaxes=True, subplot_titles=series
        )
        x = ts_index
        row = 1
        if observed is not None:
            fig.append_trace(
                go.Scatter(x=x, y=observed, name="Original"), row=row, col=1
            )
            row += 1
        if trend is not None:
            fig.append_trace(
                go.Scatter(x=x, y=trend, name="Trend"), row=row, col=1
            )
            row += 1
        if seasonal is not None:
            fig.append_trace(
                go.Scatter(x=x, y=seasonal, name="Seasonal"),
                row=row,
                col=1,
            )
            row += 1
        if resid is not None:
            fig.append_trace(
                go.Scatter(x=x, y=resid, name="Residual"), row=row, col=1
            )
            row += 1

        fig.update_layout(
            title_text="Seasonal Decomposition",
            autosize=False,
            width=1200,
            height=700,
            title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
            titlefont={"size": 20},
            legend_title=None,
            showlegend=False,
            legend=dict(
                font=dict(size=15),
                orientation="h",
                yanchor="bottom",
                y=0.98,
                xanchor="right",
                x=1,
            ),
            yaxis=dict(
                titlefont=dict(size=15),
                tickfont=dict(size=15),
            ),
            xaxis=dict(
                titlefont=dict(size=15),
                tickfont=dict(size=15),
            )
        )
        return fig

In [16]:
from statsmodels.tsa.seasonal import seasonal_decompose

res = seasonal_decompose(df['#Passengers'], period=12, model="additive", extrapolate_trend="freq")

In [17]:
fig = decomposition_plot(df.index, res.observed, res.seasonal, res.trend, res.resid)
fig.show()

La descomposición de la serie muestra claramente los distintos componentes que explican su comportamiento. La serie original presenta una tendencia creciente a lo largo del tiempo, lo cual se confirma en el componente de tendencia, que aumenta de forma progresiva. El componente estacional refleja un patrón repetitivo muy marcado que se mantiene constante en el tiempo, lo que indica una fuerte estacionalidad anual. Por último, los residuos muestran variaciones más irregulares y de menor magnitud, lo que sugiere que la mayor parte de la estructura de la serie queda explicada por la tendencia y la estacionalidad.